# 00 - Setup 

An example set of setup commands to run in a new Snowflake environment.


## Set constants 

In [ ]:
%%sql -r constants
-- Configuration variables (update these to customize the setup)
SET DATABASE_NAME = 'SF_APPLIED_AI_ML';
SET ENV_DATABASE_NAME = 'SFDEMO_ENV';
SET WAREHOUSE_NAME = 'COMPUTE_WH';
SET ROLE_NAME = 'ACCOUNTADMIN';
SET GITHUB_USER = 'sfc-gh-jmarciszewski';
SET GITHUB_PREFIX = 'https://github.com/sfc-gh-jmarciszewski';
SET GIT_SCHEMA_NAME = 'GIT';
SET NETWORK_SCHEMA_NAME = 'NETWORK_RULES';
SET SERVICE_ROLE_NAME = 'SVC_GITHUB_ROLE';
SET SERVICE_USER_NAME = 'SVC_GITHUB';
SET COMPUTE_POOL_NAME = 'SF_APPLIED_AI_ML_NOTEBOOK_COMPUTE_POOL';

## Set user role permissions for this notebook

In [ ]:
%%sql -r dataframe_12
USE ROLE IDENTIFIER($ROLE_NAME);

## Enable Cortex

https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference

In [ ]:
%%sql -r dataframe_3
-- ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

## Check network policies

In [ ]:
%%sql -r check_network
CALL snowflake.trust_center.execute_scanner('SECURITY_ESSENTIALS', 'SECURITY_ESSENTIALS_CIS3_1');

## Create default database

In [ ]:
%%sql -r create_database
CREATE DATABASE IF NOT EXISTS IDENTIFIER($DATABASE_NAME);

## Create Github service user and integration

In [ ]:
%%sql -r dataframe_1
USE DATABASE IDENTIFIER($ENV_DATABASE_NAME);

-- 0) Create schema if not exists already
CREATE SCHEMA IF NOT EXISTS IDENTIFIER($GIT_SCHEMA_NAME);

-- 1) Create a role for GitHub-driven workloads
CREATE ROLE IF NOT EXISTS IDENTIFIER($SERVICE_ROLE_NAME);

-- 2) Create a service user (TYPE=SERVICE cannot have password properties)
CREATE USER IF NOT EXISTS IDENTIFIER($SERVICE_USER_NAME)
  TYPE = SERVICE
  DEFAULT_ROLE       = SVC_GITHUB_ROLE
  DEFAULT_WAREHOUSE  = COMPUTE_WH
  DEFAULT_NAMESPACE  = SFDEMO_ENV.GIT;

-- 3) Grant role to the service user
GRANT ROLE IDENTIFIER($SERVICE_ROLE_NAME) TO USER IDENTIFIER($SERVICE_USER_NAME);

-- 4) Grant the service role access to the database, schema, and warehouse
GRANT USAGE ON DATABASE IDENTIFIER($ENV_DATABASE_NAME) TO ROLE IDENTIFIER($SERVICE_ROLE_NAME);
GRANT USAGE ON SCHEMA IDENTIFIER($GIT_SCHEMA_NAME) TO ROLE IDENTIFIER($SERVICE_ROLE_NAME);
GRANT USAGE ON WAREHOUSE IDENTIFIER($WAREHOUSE_NAME) TO ROLE IDENTIFIER($SERVICE_ROLE_NAME);

-- 5) Create the API integration for GitHub (OAuth via Snowflake GitHub App)
CREATE IF NOT EXISTS API INTEGRATION github_api_integration
  API_PROVIDER            = git_https_api
  API_ALLOWED_PREFIXES    = ($GITHUB_PREFIX)
  API_USER_AUTHENTICATION = (TYPE = SNOWFLAKE_GITHUB_APP)
  ENABLED                 = TRUE
  COMMENT                 = 'GitHub integration for sfdemo-env private repo';

-- 6) Grant the service role usage on the integration
GRANT USAGE ON INTEGRATION github_api_integration TO ROLE IDENTIFIER($SERVICE_ROLE_NAME);

-- -- 7) Create the Git repository clone (update ORIGIN to your repo URL)
-- CREATE OR REPLACE GIT REPOSITORY SFDEMO_ENV.GIT.SFDEMO_ENV_REPO
--   API_INTEGRATION = github_api_integration
--   ORIGIN          = 'https://github.com/' || $GITHUB_USER || '/<YOUR-REPO-NAME>.git';

-- manually create via UI, Workspaces > new workspace > git

## Create External Access Integration

In [ ]:
%%sql -r dataframe_4
USE DATABASE IDENTIFIER($ENV_DATABASE_NAME);

In [ ]:
%%sql -r dataframe_8
GRANT ALL PRIVILEGES ON DATABASE IDENTIFIER($ENV_DATABASE_NAME) TO ROLE IDENTIFIER($ROLE_NAME);

In [ ]:
%%sql -r dataframe_7
CREATE SCHEMA IF NOT EXISTS IDENTIFIER($NETWORK_SCHEMA_NAME);
USE SCHEMA IDENTIFIER($NETWORK_SCHEMA_NAME);

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE NETWORK RULE pypi_network_rule 
    MODE = EGRESS 
    TYPE = HOST_PORT 
    VALUE_LIST = ('pypi.org', 'files.pythonhosted.org');

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION pypi_access_integration
    ALLOWED_NETWORK_RULES = (pypi_network_rule)
    ENABLED = TRUE;

In [ ]:
%%sql -r dataframe_9
GRANT USAGE ON INTEGRATION pypi_access_integration TO ROLE IDENTIFIER($ROLE_NAME);

## Create compute pool for notebooks



In [ ]:
%%sql -r dataframe_10
CREATE COMPUTE POOL IF NOT EXISTS IDENTIFIER($COMPUTE_POOL_NAME)
  MIN_NODES = 1
  MAX_NODES = 1
  INSTANCE_FAMILY = CPU_X64_S
  AUTO_RESUME = TRUE
  AUTO_SUSPEND_SECS = 3600
  INITIALLY_SUSPENDED = TRUE
  COMMENT = 'Compute pool for running notebooks';

GRANT USAGE ON COMPUTE POOL IDENTIFIER($COMPUTE_POOL_NAME) TO ROLE IDENTIFIER($ROLE_NAME);

## Create notebook service [TODO]

In [ ]:
-- import requests
-- import json

-- token = session.connection.rest.token
-- host = session.connection._conn._rest._host

-- workspace_fqn = 'USER$.PUBLIC."snowflake-applied-ai-ml"'
-- notebook_path = '/00_env_setup/00_setup.ipynb'

-- url = f"https://{host}/api/v2/workspaces/{workspace_fqn}/notebooks/{notebook_path}/services"

-- payload = {
--     "action": "create_and_connect_default_service",
--     "compute_type": "CPU"
-- }

-- headers = {
--     "Authorization": f"Snowflake Token=\"{token}\"",
--     "Content-Type": "application/json"
-- }

-- response = requests.post(url, headers=headers, json=payload)
-- print(response.status_code)
-- print(response.json())

# APPENDIX

In [ ]:
-- ALTER ACCOUNT SET DEFAULT_NOTEBOOK_COMPUTE_POOL_CPU = 'SF_APPLIED_AI_ML_NOTEBOOK_COMPUTE_POOL';
-- ALTER ACCOUNT SET DEFAULT_NOTEBOOK_EAI_CPU = ('pypi_access_integration');